In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import numpy as np
import pandas as pd

## Hugging Face Feature Extractor Shape Task: 


Initialize Hugging Face's AutoFeatureExtractor using the "MIT/ast-finetuned-audioset-10-10-0.4593" checkpoint. 

To test your pipeline, create a dummy audio mix of exactly 160,000 ones using 
mix = np.ones(160000). 
Pass this dummy array into the extractor with sampling_rate=16000 and return_tensors="pt". 
Finally, extract the input_values from the resulting dictionary and apply .squeeze(0) to remove the default batch dimension. 


Question: What is the exact shape of the resulting PyTorch tensor? (This is the spectrogram matrix the AST model will process). In this format: [a,b] or (a,b)

In [2]:
from transformers import AutoFeatureExtractor

extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

mix = np.ones(160000)

inputs = extractor(
    mix,
    sampling_rate=16000,
    return_tensors="pt"
)

spectrogram = inputs["input_values"].squeeze(0)

# 5. Print shape
print(spectrogram.shape)

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

torch.Size([1024, 128])


## Model Architecture Initialization Task: 
Initialize the ASTForAudioClassification.from_pretrained model using the "MIT/ast-finetuned-audioset-10-10-0.4593" checkpoint. 

Because we are classifying 10 genres instead of the original AudioSet classes, you must explicitly pass num_labels=10 and ignore_mismatched_sizes=True to replace the classification head. 

Question: Write a quick loop using 

sum(p.numel() for p in model.parameters() if p.requires_grad) 

to calculate the number of trainable parameters in this newly configured model. What is the exact integer value?

In [3]:
from transformers import ASTForAudioClassification

model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(trainable_params)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


86196490


## Inference Normalization Math Task: In your test inference loop, right before feature extraction, the audio is normalized to prevent clipping using this exact formula: 

y = y / (np.max(np.abs(y)) + 1e-9). 

To test this, create a NumPy array: 
y_test = np.array([-0.85, 0.40, 0.20, -0.10]). 

Apply the normalization formula to it. 

Question: What is the resulting value at index 0 of the normalized array, rounded to 3 decimal places?

In [5]:
y = np.array([-0.85, 0.40, 0.20, -0.10])

y = y / (np.max(np.abs(y)) + 1e-9)

print(round(y[0], 3))

-1.0
